In [2]:
import sys
from pathlib import Path
import os
import tifffile as tiff
import numpy as np

project_root = Path().resolve().parent
sys.path.append(str(project_root / "src"))

In [41]:
## To reload

import importlib
import pinn.visual as visual
import pinn.movie as movie
importlib.reload(visual)
importlib.reload(movie)

<module 'pinn.movie' from '/mmfs1/gscratch/matsulab/sim/pinn-exploration-model/src/pinn/movie.py'>

In [3]:
## INPUT DATA

import mrcfile
import os
import numpy as np
from flax.training import checkpoints

# ==== Parameters =====
lambda_1 = 100000
lambda_2 = 1
lambda_3 = 100000
lambda_4 = 0.01
step = 10000
hidden_dim = 128
phase = 2
axis = "x"

# ==== Parameters for Golgi =====
data_id = "czii_27042022"
shape = "golgi_01"
output_path = "../outputs/figs/phase/golgi"
N_voxel = 512
voxel_size = 0.784 # nm
half_length = N_voxel * voxel_size / 2.0
x_range = (-0.4, 0.38) # this is written in scale-less unit
y_range = (-0.9, 1.0)
z_range = (-0.9, 0.9)

# # ==== Parameters for Mito =====
# data_id = "mendelsohn_2021"
# shape = "mito_01"

# ==== Load checkpoint data =====
checkpoint_dir = f"../outputs/logs/{data_id}/{shape}_{hidden_dim}/phase{phase}_{lambda_1}_{lambda_2}_{lambda_3}_{lambda_4}"
checkpoint_path = os.path.abspath(f"{checkpoint_dir}/checkpoint_{step}")
checkpoint_data = checkpoints.restore_checkpoint(ckpt_dir=checkpoint_path, target=None)

# ===== Load cryoET data =====
if shape == "golgi_01":
    mrc_file_path = f"../data/experimental/vol/formatted/{data_id}/{shape}.mrc"
elif shape == "mito_01":
    mrc_file_path = f"../data/experimental/vol/raw/{data_id}/{shape}.mrc"
# with mrcfile.open(mrc_file_path, permissive=True) as mrc:
    # cryoET_data = mrc.data

In [33]:
### PHASE FIELD VTI DATA ###
# This makes golgi_phase_field.vti (volumetric phase field data)

from pinn.movie import build_phi_fn
from pinn.visual import evaluate_phase_field_for_vtk, numpy_phase_field_to_vtk_image, save_phase_field_and_surface
from pinn.model import PINN

model = PINN(hidden_dim=hidden_dim)
params = checkpoint_data["state"]["params"]

def phi_fn(points):
    return model.apply(params, points)

phi_volume, coordinates = evaluate_phase_field_for_vtk(
    phi_fn=phi_fn,
    grid_size=64,
    x_range=x_range,
    y_range=y_range,
    z_range=z_range,
    transpose=True,
    batch_size=100_000,
)

vtk_image = numpy_phase_field_to_vtk_image(
    phi_volume,
    coordinates,
    physical_scale=half_length,
)

save_phase_field_and_surface(
    vtk_image,
    output_prefix=f"{output_path}",
    iso_value=0.0,
)

W0806 17:39:21.196699   84572 gemm_fusion_autotuner.cc:1050] Compiling 4 configs for gemm_fusion_dot_general.1 on a single thread.
W0806 17:39:26.247789   84572 gemm_fusion_autotuner.cc:1050] Compiling 9 configs for gemm_fusion_dot_general.1 on a single thread.
W0806 17:39:28.437304   84572 gemm_fusion_autotuner.cc:1050] Compiling 12 configs for gemm_fusion_dot.1 on a single thread.
W0806 17:39:30.164284   84572 gemm_fusion_autotuner.cc:1050] Compiling 4 configs for gemm_fusion_dot_general.1 on a single thread.
W0806 17:39:30.715420   84572 gemm_fusion_autotuner.cc:1050] Compiling 9 configs for gemm_fusion_dot_general.1 on a single thread.


Saved phase field: /mmfs1/gscratch/matsulab/sim/pinn-exploration-model/outputs/figs/phase/golgi_phase_field.vti
Saved surface:     /mmfs1/gscratch/matsulab/sim/pinn-exploration-model/outputs/figs/phase/golgi_surface.vtp
Surface contains 35487 points and 69285 triangles.


('/mmfs1/gscratch/matsulab/sim/pinn-exploration-model/outputs/figs/phase/golgi_phase_field.vti',
 '/mmfs1/gscratch/matsulab/sim/pinn-exploration-model/outputs/figs/phase/golgi_surface.vtp')

In [4]:
### ORIGINAL INPUT VTI DATA ###
# This makes golgi_input_volume.vti (VTI version of input MRC data)

from pinn.visual import mrc_to_vti_like_phase_field

with mrcfile.open(mrc_file_path, permissive=True) as mrc:
    mrc_shape = mrc.data.shape

mrc_vtk_image, cropped_mrc, coordinates = (
    mrc_to_vti_like_phase_field(
        mrc_path=mrc_file_path,
        vti_path=f"{output_path}_input_volume.vti",
        x_range=x_range,
        y_range=y_range,
        z_range=z_range,
        transpose=True,
        half_length=half_length,
    )
)


Raw MRC shape: (128, 128, 128)
Oriented shape (z,y,x): (128, 128, 128)
Crop ranges:
  x: (-0.4, 0.38)
  y: (-0.9, 1.0)
  z: (-0.9, 0.9)
Crop slices:
  x: [39:88]
  y: [7:128]
  z: [7:121]
Cropped shape (z,y,x): (114, 121, 49)
VTK dimensions (x,y,z): (49, 121, 114)
VTK origin: (-77.43697637795276, -178.5791496062992, -178.5791496062992)
VTK spacing: (3.16069291338583, 3.160692913385816, 3.160692913385816)
VTK bounds: (-77.43697637795276, 74.27628346456709, -178.5791496062992, 200.7039999999987, -178.5791496062992, 178.57914960629796)
Saved: ../outputs/figs/phase/golgi_input_volume.vti


In [36]:
mrc_shape

(128, 128, 128)